<a href="https://colab.research.google.com/github/LEF9701/CUDA/blob/main/libtorch_basic.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!cmake --version
!gcc --version | head -n 1
!lsb_release -d

cmake version 3.31.10

CMake suite maintained and supported by Kitware (kitware.com/cmake).
gcc (Ubuntu 11.4.0-1ubuntu1~22.04.3) 11.4.0
Description:	Ubuntu 22.04.5 LTS


Download and extract LibTorch (CPU)

In [2]:
!wget -q https://download.pytorch.org/libtorch/cpu/libtorch-cxx11-abi-shared-with-deps-latest.zip
!unzip -q -o libtorch-cxx11-abi-shared-with-deps-latest.zip
print("LibTorch downloaded and extracted.")

LibTorch downloaded and extracted.


In [3]:
%%writefile main.cpp
#include <torch/torch.h>
#include <iostream>

int main() {
    std::cout << "=== Welcome to the C++ side of MLSys ===" << std::endl;

    // 1. Create a 2x3 random tensor using LibTorch
    torch::Tensor x = torch::rand({2, 3});
    std::cout << "Tensor x:\n" << x << std::endl;

    // 2. Create a 3x2 tensor of ones
    torch::Tensor y = torch::ones({3, 2});
    std::cout << "Tensor y:\n" << y << std::endl;

    // 3. Matrix multiplication at the C++ level (2x3 * 3x2 = 2x2)
    torch::Tensor result = torch::matmul(x, y);
    std::cout << "Matrix multiplication result (x * y):\n" << result << std::endl;

    return 0;
}

Writing main.cpp


In [4]:
%%writefile CMakeLists.txt
cmake_minimum_required(VERSION 3.18 FATAL_ERROR)
project(mlsys_demo)

# Point CMake at the LibTorch we extracted in Step 1
list(APPEND CMAKE_PREFIX_PATH "${CMAKE_CURRENT_SOURCE_DIR}/libtorch")
find_package(Torch REQUIRED)
set(CMAKE_CXX_FLAGS "${CMAKE_CXX_FLAGS} ${TORCH_CXX_FLAGS}")

# Build the executable
add_executable(mlsys_demo main.cpp)
target_link_libraries(mlsys_demo "${TORCH_LIBRARIES}")
set_property(TARGET mlsys_demo PROPERTY CXX_STANDARD 17)

Writing CMakeLists.txt


In [5]:
!rm -rf build
!mkdir -p build
%cd build
!cmake .. -DCMAKE_BUILD_TYPE=Release
!make -j$(nproc)
!./mlsys_demo
%cd ..

/content/build
-- The C compiler identification is GNU 11.4.0
-- The CXX compiler identification is GNU 11.4.0
-- Detecting C compiler ABI info
-- Detecting C compiler ABI info - done
-- Check for working C compiler: /usr/bin/cc - skipped
-- Detecting C compile features
-- Detecting C compile features - done
-- Detecting CXX compiler ABI info
-- Detecting CXX compiler ABI info - done
-- Check for working CXX compiler: /usr/bin/c++ - skipped
-- Detecting CXX compile features
-- Detecting CXX compile features - done
-- Performing Test CMAKE_HAVE_LIBC_PTHREAD
-- Performing Test CMAKE_HAVE_LIBC_PTHREAD - Success
-- Found Threads: TRUE
CMake Warning (dev) at /usr/local/lib/python3.12/dist-packages/cmake/data/share/cmake-3.31/Modules/FindPackageHandleStandardArgs.cmake:441 (message):
  The package name passed to `find_package_handle_standard_args` (torch) does
  not match the name of the calling package (Torch).  This can lead to
  problems in calling code that expects `find_package` result 